In [ ]:
#|default_exp accel

# Load deps

In [ ]:
# !pip install -q torcheval diffusers accelerate

In [ ]:
# # # if src modules imported
# # from google.colab import drive
# # drive.mount('/content/drive')
# import sys
# app_path = '/content/drive/MyDrive/Projects/miniSD'
# sys.path.append(app_path)

In [ ]:
#|export
import torch
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from functools import partial

import torchvision.transforms.functional as TF
import torch.nn.functional as F
from torch import nn,optim
from torch.nn import init
from torch.optim import lr_scheduler
from diffusers import UNet2DModel
from datasets import load_dataset
from torch.utils.data import DataLoader, default_collate
from accelerate import Accelerator

from src.utils import set_seed, store_attr
from src.datasets import inplace, DataLoaders, show_images
from src.learner import (
    SingleBatchCB, Callback, DeviceCB,
    TrainLearner, ProgressCB, MetricsCB,
    CompletionCB, TrainCB, Learner
)
from src.sgd import BatchSchedCB
from src.conv import def_device
from src.init import clean_mem

# Config

In [ ]:
torch.set_printoptions(precision=2, linewidth=140, sci_mode=False)
set_seed(42)
mpl.rcParams['image.cmap'] = 'gray_r'
ds_name = "fashion_mnist"
ds_xl,ds_yl = 'image','label'
bs = 128
num_dl_workers = 2
lr = 5e-3
epochs = 3


# diffusion pipeline
betamin,betamax,n_steps = 0.0001,0.02,1000
beta = torch.linspace(betamin, betamax, n_steps)
alpha = 1.-beta
alphabar = alpha.cumprod(dim=0)
sigma = beta.sqrt()

# Load data

In [ ]:
@inplace
def transformi(b): b[ds_xl] = [F.pad(TF.to_tensor(o), (2,2,2,2)) for o in b[ds_xl]]
# a simpler way to convert images from 28x28 to 32x32 
dsd = load_dataset(ds_name)
tds = dsd.with_transform(transformi)
dls = DataLoaders.from_dd(tds, bs, num_workers=num_dl_workers)

In [ ]:
dt = dls.train
xb,yb = next(iter(dt))
print(xb.shape)
print(yb[:10])
show_images(xb[:16], figsize=(5,5));

# Plot the noise schedule

In [ ]:
fig, axs = plt.subplots(1,3, figsize=(15,3))
axs[0].plot(beta);
axs[0].set_title("beta");
axs[1].plot(sigma);
axs[1].set_title("sigma");
axs[2].plot(alphabar);
axs[2].set_title("alphabar");

# Define DDPM components

## Noisifier

In [ ]:
def noisify(x0, ᾱ):
    device = x0.device
    n = len(x0)
    t = torch.randint(0, n_steps, (n,), dtype=torch.long)
    ε = torch.randn(x0.shape, device=device)
    ᾱ_t = ᾱ[t].reshape(-1, 1, 1, 1).to(device)
    xt = ᾱ_t.sqrt()*x0 + (1-ᾱ_t).sqrt()*ε
    return (xt, t.to(device)), ε

In [ ]:
(xt,t),ε = noisify(xb[:25],alphabar)
print(t)
print("=================")
show_images(xt, imsize=1.5, titles=list(map(str, t.data.numpy())));

## Sampler

In [ ]:
@torch.no_grad()
def sample(learn, sz, alpha, alphabar, sigma, n_steps):
    device = next(learn.model.parameters()).device
    x_t = torch.randn(sz, device=device)
    preds = []
    for t in reversed(range(n_steps)):
        t_batch = torch.full((x_t.shape[0],), t, device=device, dtype=torch.long)
        z = (torch.randn(x_t.shape) if t > 0 else torch.zeros(x_t.shape)).to(device)
        ᾱ_t1 = alphabar[t-1]  if t > 0 else torch.tensor(1)
        b̄_t = 1 - alphabar[t]
        b̄_t1 = 1 - ᾱ_t1
        x_0_hat = ((x_t - b̄_t.sqrt() * learn.model((x_t, t_batch)))/alphabar[t].sqrt()).clamp(-1,1)
        x_t = x_0_hat * ᾱ_t1.sqrt()*(1-alpha[t])/b̄_t + x_t * alpha[t].sqrt()*b̄_t1/b̄_t + sigma[t]*z
        preds.append(x_t.cpu())
    return preds

## Define DDPM CB without using `TrainCB`
- we don't have access to the `predict` method of `TrainCB` here.
- instead, we use another trick to modify the forward method of the model itself

In [ ]:
class DDPMCB(Callback):
    order = DeviceCB.order+1
    def __init__(self, n_steps, beta_min, beta_max):
        super().__init__()
        store_attr()
        self.beta = torch.linspace(self.beta_min, self.beta_max, self.n_steps)
        self.α = 1. - self.beta 
        self.ᾱ = torch.cumprod(self.α, dim=0)
        self.σ = self.beta.sqrt()
    
    def before_batch(self, learn): learn.batch = noisify(learn.batch[0], self.ᾱ)
    def sample(self, learn, sz): return sample(learn, sz, self.α, self.ᾱ, self.σ, self.n_steps)
    
class UNet(UNet2DModel):
    def forward(self, x): return super().forward(*x).sample

In [ ]:
ddpm_cb = DDPMCB(n_steps=n_steps, beta_min=betamin, beta_max=betamax)
model = UNet(
    in_channels=1, out_channels=1,
    block_out_channels=(16, 32, 64, 64),
    # smaller channel sizes by a factor of 2 --> make things faster
    norm_num_groups=8
    # number of groups for GroupNorm. all channel sizes must be divisible by this
)
learn = TrainLearner(model, dls, nn.MSELoss())
learn.fit(train=False, cbs=[ddpm_cb,SingleBatchCB(), CompletionCB()])
(xt,t),ε = learn.batch
titles = list(map(str, t[:25].data.numpy()))
show_images(xt[:25], titles=titles, imsize=1.5);

## Modify the UNET initiation (default is not optimal!!!)

In [ ]:
def init_ddpm(model):
    for o in model.down_blocks:
        for p in o.resnets:
            p.conv2.weight.data.zero_()
            # this is to make sure the Skip/Residual path does nothing at first
            # similar to what we had in our _conv_block in ResNet Block definition
            if o.downsamplers:
                for p in list(o.downsamplers): init.orthogonal_(p.conv.weight)
                # TODO: how does this work? what are downsamplers?

    for o in model.up_blocks:
        for p in o.resnets: p.conv2.weight.data.zero_()
        # TODO: what is the difference between down_blocks and up_blocks?

    model.conv_out.weight.data.zero_()
    # this makes the model to predict zero as noise at the beginning
    # TODO: why does this help?

# Train

In [ ]:
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [ddpm_cb, DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched)]
model = UNet(in_channels=1, out_channels=1, block_out_channels=(16, 32, 64, 128), norm_num_groups=8)
init_ddpm(model)
opt_func = partial(optim.Adam, eps=1e-5)
# the default Adam eps=1e-8
# we divide by this value to avoid division by zero.
# if it's too small, it makes the effective `lr` too huge.
learn = TrainLearner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)
# learn.fit(epochs)

In [ ]:
mdl_path = Path('models')
mdl_path.mkdir(exist_ok=True)
# torch.save(learn.model, mdl_path/'fashion_ddpm2.pkl')
# learn.model = torch.load(mdl_path/'fashion_ddpm2.pkl', weights_only=False)

# Sampling from trained model

In [ ]:
# samples = ddpm_cb.sample(learn, (16, 1, 32, 32))
# show_images(samples[-1], figsize=(5,5));

# How to make training faster?

## Mixed Precision

### Implement DDPM without using callback

- put the noisifier inside the data collator function

In [ ]:
def collate_ddpm(b): return noisify(default_collate(b)[ds_xl], alphabar)
def dl_ddpm(ds): return DataLoader(
    ds, batch_size=bs * 4, # a larger batch size to keep gpu busy
    collate_fn=collate_ddpm, num_workers=num_dl_workers
)
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

In [ ]:
#|export
class MixedPrecision(TrainCB):
    # this callback replaces the common with torch.autocast(...): approach
    order = DeviceCB.order+10
    
    def before_fit(self, learn): self.scaler = torch.amp.GradScaler(def_device)

    def before_batch(self, learn):
        self.autocast = torch.autocast("cuda", dtype=torch.float16)
        self.autocast.__enter__() # enter the context manager

    def after_loss(self, learn):
        self.autocast.__exit__(None, None, None) # exit the context manager
        
    # we have to use the `Learner` class to access the following methods by __getattr__ method
    def backward(self, learn): self.scaler.scale(learn.loss).backward()
    def step(self, learn):
        self.scaler.step(learn.opt)
        self.scaler.update()

In [ ]:
clean_mem()

In [ ]:
# greater batch size means we have less opportunites to updated parameters
# but we can increase the learning rate
# So, in order to achieve the same performance we increase the 'lr' and 'epochs'
# TODO: does this prevent getting stuck in sharp local minima as achieved by small `bs` and `lr`
lr = 1e-2
epochs = 6
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
cbs = [DeviceCB(), MixedPrecision(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched)]
model = UNet(in_channels=1, out_channels=1, block_out_channels=(16, 32, 64, 128), norm_num_groups=8)
init_ddpm(model)
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)
# learn.fit(epochs)

In [ ]:
# samples = sample(learn, (32, 1, 32, 32), alpha, alphabar, sigma, n_steps)
# show_images(samples[-1][:25], imsize=1.5);

In [ ]:
# torch.save(learn.model, 'models/fashion_ddpm_mp.pkl')

## Accelerate

In [ ]:
#|export
class AccelerateCB(TrainCB):
    # here, we're only using accelerate as a cleaner alternative to `MixedPrecision` callback
    # it handles AMP seamlessly
    # it also can be used to run distributed train on GPUs and TPUs
    order = DeviceCB.order+10
    def __init__(self, n_inp=1, mixed_precision="fp16"):
        super().__init__(n_inp=n_inp)
        self.acc = Accelerator(mixed_precision=mixed_precision)
        
    def before_fit(self, learn):
        learn.model,learn.opt,learn.dls.train,learn.dls.valid = self.acc.prepare(
            learn.model, learn.opt, learn.dls.train, learn.dls.valid)

    def backward(self, learn): self.acc.backward(learn.loss)

In [ ]:
def noisify(x0, ᾱ):
    device = x0.device
    n = len(x0)
    t = torch.randint(0, n_steps, (n,), dtype=torch.long)
    ε = torch.randn(x0.shape, device=device)
    ᾱ_t = ᾱ[t].reshape(-1, 1, 1, 1).to(device)
    xt = ᾱ_t.sqrt()*x0 + (1-ᾱ_t).sqrt()*ε
    # we return a tuple of 3 tensors instead of tuple of tuples
    # we achieve the same fucntionality by using TrainCB `n_inp` argument
    return xt, t.to(device), ε

In [ ]:
dls = DataLoaders(dl_ddpm(tds['train']), dl_ddpm(tds['test']))

In [ ]:
# a much cleaner and more modular definition of DDMP CB
class DDPMCB2(Callback):
    def after_predict(self, learn): learn.preds = learn.preds.sample

In [ ]:
model = UNet2DModel(in_channels=1, out_channels=1, block_out_channels=(16, 32, 64, 128), norm_num_groups=8)
init_ddpm(model)
cbs = [DDPMCB2(), DeviceCB(), ProgressCB(plot=True), MetricsCB(), BatchSchedCB(sched), AccelerateCB(n_inp=2)]
learn = Learner(model, dls, nn.MSELoss(), lr=lr, cbs=cbs, opt_func=opt_func)
learn.fit(epochs)

## A sneaky trick

- in some cases where we have powerful/multiple GPUs but small number of CPUs, it's hard to keep the GPU busy
- this is the case in kaggle for example where we have T4x2 GPUs but we only have 2 CPUs!!!
- from the training POV, looking at the same batch 2-4 times in a row is totally fine
- So, we can load a batch and serve it multiple times (as many times as the number of GPUs is a good practice)

In [ ]:
#|export
class MultDL:
    def __init__(self, dl, mult=2): self.dl,self.mult = dl,mult
    def __len__(self): return len(self.dl)*self.mult
    def __iter__(self):
        # it loads and augments the data once (each batch) (CPU work)
        for o in self.dl:
            # returns the same prepared batch multiple times
            # TODO: can we config this to serve the same batch once for each GPU?? using accelerate??
            for i in range(self.mult): yield o

In [ ]:
dls.train = MultDL(dls.train)